# The Well · `active_matter` — statistical analysis

This notebook analyzes how **initial / control parameters** (`alpha` = dipole strength, `zeta` = alignment) drive simulation outcomes, validates approximate physics laws, and flags anomalous trajectories.

**Data:** tidy feature table built by `python -m src.extract_features` (synthetic demo by default; stream real HDF5 from Hugging Face when ready).

**Methods:** two-way ANOVA, one-way ANOVA + Tukey, pairwise t-tests, assumption checks, Kruskal–Wallis fallback, physics checks, MAD/IQR anomaly detection.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.stats import (
    assumption_checks,
    detect_anomalies,
    kruskal_wallis,
    one_way_anova,
    pairwise_ttests,
    physics_validation,
    two_way_anova,
)

sns.set_theme(style="whitegrid", context="notebook")
FEATURES = ROOT / "data" / "features.parquet"
df = pd.read_parquet(FEATURES)
print(df.shape)
df.head()

## 1. Exploratory data analysis

Each row is one trajectory. Factors: `alpha` ∈ {-1,…,-5}, `zeta` ∈ {1,3,…,17}. Primary response for the phase transition: `nematic_order_S`.

In [ ]:
print(df.groupby(["alpha", "zeta"]).size().unstack(fill_value=0))
print("\nFeature summary:")
df[["nematic_order_S", "kinetic_energy", "enstrophy", "mean_concentration", "div_u_rms"]].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x="zeta", y="nematic_order_S", ax=axes[0], color="#1B9AAA")
axes[0].set_title("Nematic order by alignment zeta")
sns.boxplot(data=df, x="alpha", y="nematic_order_S", ax=axes[1], color="#E76F51")
axes[1].set_title("Nematic order by dipole alpha")
plt.tight_layout()
plt.show()

g = sns.catplot(
    data=df, kind="point", x="zeta", y="nematic_order_S",
    hue="alpha", errorbar="sd", height=4, aspect=1.6
)
g.fig.suptitle("Interaction preview: S(zeta) by alpha", y=1.02)
plt.show()

## 2. Two-way ANOVA — how much do initial circumstances matter?

Model: `nematic_order_S ~ C(alpha) * C(zeta)`.

We report **F**, **p**, and **partial η²** (effect size) for each term. Large partial η² for `zeta` is expected if the isotropic→nematic transition is alignment-driven.

In [ ]:
tw = two_way_anova(df, "nematic_order_S", "alpha", "zeta")
display(tw["table"])
print("Verdict:", tw["verdict"])
print("N =", tw["n"])

## 3. One-way ANOVA + Tukey HSD + pairwise t-tests

Focus on `zeta` (the strongest expected driver) and compare groups.

In [ ]:
ow = one_way_anova(df, "nematic_order_S", "zeta")
print(f"F={ow['f']:.3f}, p={ow['p']:.3e}, eta^2={ow['eta_sq']:.3f}, omega^2={ow['omega_sq']:.3f}")
print("SSE (SSW) =", ow["ss_within"])
print("Verdict:", ow["verdict"])
print("\nTukey HSD (head):")
display(ow["tukey"].head(12))

tt = pairwise_ttests(df, "nematic_order_S", "zeta")
print("\nPairwise Welch t-tests (Holm-adjusted), head:")
display(tt.head(12))

## 4. Assumption checks & nonparametric fallback

ANOVA assumes roughly equal variances (Levene) and normal residuals (Shapiro). If violated, use Kruskal–Wallis.

In [ ]:
assumptions = assumption_checks(df, "nematic_order_S", "zeta")
display(pd.Series(assumptions))

if assumptions["recommend_nonparametric"]:
    kw = kruskal_wallis(df, "nematic_order_S", "zeta")
    print("Kruskal-Wallis H={H:.3f}, p={p:.3e}, k={k}, n={n}".format(**kw))
else:
    print("Parametric ANOVA assumptions look acceptable.")

## 5. Physics-law validation

1. **Concentration conservation** — mean concentration ≈ 1 (periodic BCs, initialized at c=1).
2. **Near-incompressibility** — Stokes fluid ⇒ ∇·u ≈ 0 (we use RMS of discrete divergence).
3. **Phase transition** — nematic order should increase with alignment `zeta` (Spearman correlation).

In [ ]:
phys = physics_validation(df)
for name, block in phys.items():
    print(f"[{name}] {block['message']}")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].hist(df["mean_concentration"], bins=30, color="#1B9AAA", edgecolor="white")
axes[0].axvline(1.0, color="red", ls="--", label="target=1")
axes[0].set_title("mean concentration"); axes[0].legend()
axes[1].hist(df["div_u_rms"], bins=30, color="#E76F51", edgecolor="white")
axes[1].set_title("div_u_rms")
sns.scatterplot(data=df, x="zeta", y="nematic_order_S", hue="alpha", ax=axes[2], palette="viridis")
axes[2].set_title("S vs zeta")
plt.tight_layout(); plt.show()

## 6. Anomaly detection within (alpha, zeta) cells

For each parameter cell, flag trajectories whose response is an outlier (robust MAD z-score > 3.5 by default). Useful for spotting atypical random initializations or corrupted downloads.

In [ ]:
flagged = detect_anomalies(
    df, response="nematic_order_S", group_cols=["alpha", "zeta"], method="mad", threshold=3.5
)
n_flag = int(flagged["is_anomaly"].sum())
print(f"Flagged {n_flag} / {len(flagged)} trajectories")
cols = ["file", "traj_idx", "alpha", "zeta", "nematic_order_S", "anomaly_score"]
if "injected_anomaly" in flagged.columns:
    cols.append("injected_anomaly")
display(flagged.loc[flagged["is_anomaly"], cols].sort_values("anomaly_score", ascending=False))

plt.figure(figsize=(8, 4))
sns.scatterplot(
    data=flagged, x="zeta", y="nematic_order_S",
    hue="is_anomaly", style="alpha", palette={False: "#1B9AAA", True: "#9B2226"}
)
plt.title("Anomalies in nematic order within parameter cells")
plt.show()

## 7. Repeat ANOVA for other responses

Compare effect sizes of `alpha` / `zeta` across several physics features.

In [ ]:
responses = [
    "nematic_order_S", "kinetic_energy", "enstrophy",
    "std_concentration", "time_to_steady", "spectral_slope"
]
rows = []
for resp in responses:
    tw = two_way_anova(df, resp, "alpha", "zeta")
    table = tw["table"].set_index("source")
    for src in ["C(alpha)", "C(zeta)", "C(alpha):C(zeta)"]:
        if src in table.index:
            rows.append({
                "response": resp,
                "term": src,
                "F": table.loc[src, "F"],
                "p": table.loc[src, "p"],
                "partial_eta_sq": table.loc[src, "partial_eta_sq"],
            })
effect = pd.DataFrame(rows)
display(effect.pivot(index="response", columns="term", values="partial_eta_sq"))

plt.figure(figsize=(9, 4))
sns.barplot(data=effect, x="response", y="partial_eta_sq", hue="term")
plt.xticks(rotation=30, ha="right")
plt.ylabel("partial eta^2")
plt.title("How much do alpha / zeta / interaction explain each feature?")
plt.tight_layout(); plt.show()

## Next steps

- Launch the interactive dashboard: `streamlit run app/streamlit_app.py`
- Replace the demo table with streamed Well data:
  `python -m src.extract_features --splits train --time-stride 4 --space-stride 8`
- (Optional) Restrict to a few files while testing: `--max-files 5`